In [1]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Literal

class SentimentOutputSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(..., description="The sentiment of the review.")

sentiment_llm = llm.with_structured_output(SentimentOutputSchema)

In [5]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    assessment: dict
    response: str

In [6]:
from langchain_core.prompts import PromptTemplate

def find_sentiment(ReviewState):
    review = ReviewState['review']
    prompt_template = PromptTemplate(
        input_variables=["review"],
        template="Determine the sentiment of the following review: {review}. Respond with either 'positive' or 'negative'."
    )
    chain = prompt_template | sentiment_llm
    sentiment_result = chain.invoke({"review": review})
    return {"sentiment" : sentiment_result.sentiment}

In [7]:
graph = StateGraph(ReviewState)

In [8]:
def check_sentiment_mood(ReviewState):
    sentiment = ReviewState['sentiment']
    if sentiment == "positive":
        return 'generate_positive_response'
    else:
        return 'run_assessment'

In [ ]:
from langchain_core.output_parsers import StrOutputParser


def run_assessment(ReviewState):
    review = ReviewState['review']
    prompt_template = PromptTemplate(
        input_variables=["review"],
        template="Assess the following review: {review}. Provide a detailed analysis of the issues mentioned."
    )
    chain = prompt_template | llm
    assessment_result = chain.invoke({"review": review})
    return {"assessment": assessment_result}

def generate_positive_response(ReviewState):
    review = ReviewState['review']
    prompt_template = PromptTemplate(
        input_variables=["review"],
        template="Generate a positive response to the following review: {review}."
    )
    parser = StrOutputParser()
    chain = prompt_template | llm | parser
    response_result = chain.invoke({"review": review})
    return {"response": response_result}